In [1]:
import os
import sys

In [2]:
sys.path.append(os.path.abspath("../.."))

In [3]:
from datalemur_utils import extract_iframe_from_url

In [4]:
url = "https://datalemur.com/questions/time-spent-snaps"
extract_iframe_from_url(url)

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as Sum, round as Round, coalesce, when, lit

In [6]:
spark = SparkSession.builder.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/07 02:00:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
activities = spark.read\
.option("delimiter", "\t")\
.option("header", "true")\
.option("inferSchema", "true")\
.format("csv")\
.load("activities.txt")

In [8]:
age_breakdown = spark.read\
.option("delimiter", "\t")\
.option("header", "true")\
.option("inferSchema", "true")\
.format("csv")\
.load("age_breakdown.txt")

In [9]:
activities.createOrReplaceTempView("activities")
age_breakdown.createOrReplaceTempView("age_breakdown")

In [10]:
activities.show()

+-----------+-------+-------------+----------+-------------------+
|activity_id|user_id|activity_type|time_spent|      activity_date|
+-----------+-------+-------------+----------+-------------------+
|       7274|    123|         open|       4.5|06/22/2022 12:00:00|
|       2425|    123|         send|       3.5|06/22/2022 12:00:00|
|       1413|    456|         send|      5.67|06/23/2022 12:00:00|
|       1414|    789|         chat|      11.0|06/25/2022 12:00:00|
|       2536|    456|         open|       3.0|06/25/2022 12:00:00|
+-----------+-------+-------------+----------+-------------------+



In [11]:
age_breakdown.show()

+-------+----------+
|user_id|age_bucket|
+-------+----------+
|    123|     31-35|
|    456|     26-30|
|    789|     21-25|
+-------+----------+



In [12]:
spark.sql(
    """
    select ab.age_bucket, 
    coalesce(round((sum(case when act.activity_type = 'send' then act.time_spent end)/sum(case when act.activity_type in ('open', 'send') then act.time_spent end))*100.0, 2), 0.0) as open_prct,
    coalesce(round((sum(case when act.activity_type = 'open' then act.time_spent end)/sum(case when act.activity_type in ('open', 'send') then act.time_spent end))*100.0, 2), 0.0) as send_prct
    from activities act
    join age_breakdown ab
    on act.user_id = ab.user_id
    group by 1
    """
).show()

+----------+---------+---------+
|age_bucket|open_prct|send_prct|
+----------+---------+---------+
|     21-25|      0.0|      0.0|
|     31-35|    43.75|    56.25|
|     26-30|     65.4|     34.6|
+----------+---------+---------+



In [13]:
activities.join(age_breakdown, on="user_id", how="inner")\
.groupBy(age_breakdown.age_bucket).agg(
    coalesce(Round(Sum(
        when(activities.activity_type == "open", activities.time_spent)
    )/Sum(
        when(activities.activity_type.isin("open", "send"), activities.time_spent)
    )*100.0, 2),lit(0.0)).alias("open_prct"),

    coalesce(Round(Sum(
        when(activities.activity_type == "send", activities.time_spent)
    )/Sum(
        when(activities.activity_type.isin("open", "send"), activities.time_spent)
    )*100.0, 2),lit(0.0)).alias("send_prct")
)\
.show()

+----------+---------+---------+
|age_bucket|open_prct|send_prct|
+----------+---------+---------+
|     21-25|      0.0|      0.0|
|     31-35|    56.25|    43.75|
|     26-30|     34.6|     65.4|
+----------+---------+---------+



In [14]:
spark.stop()